# 1. Import Library

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np
import joblib
import pickle

In [2]:
df = pd.read_csv("dropout_dataset_clean.csv")
pd.set_option("display.max_columns", None)
df.head(15)

,location_type,family_income,financial_aid_status,distance_to_institute,internet_connectivity_issues,motivation_score,career_alignment,stress_levels,family_support,attendance_rate,test_scores_avg,backlogs,teaching_quality_rating,dropout
0,Urban,30681.0,1,4.60,0,5,1,1,3,77.2,67.1,1,4,0
1,Rural,17579.0,1,50.00,1,5,1,2,2,60.8,47.5,0,7,1
2,Urban,4952.0,1,5.30,2,3,1,2,1,57.2,66.2,0,8,1
3,Rural,14778.0,1,27.89,0,4,2,3,1,62.6,44.9,1,6,1
4,Semi-urban,8725.0,1,6.10,0,5,1,2,2,83.9,100.0,2,8,0
5,Rural,3480.0,0,12.61,0,8,2,3,3,55.0,78.8,1,7,0
6,Rural,12614.0,1,8.60,1,7,3,1,2,69.3,87.2,0,10,0
7,Rural,31353.0,1,17.93,0,5,2,2,2,82.4,71.8,0,7,0
8,Semi-urban,4437.0,2,7.20,1,6,3,1,3,27.9,67.4,0,8,0
9,Semi-urban,4922.0,0,8.20,1,5,2,2,3,77.9,81.3,1,10,0


In [3]:
log_features = ["family_income", "distance_to_institute"]

categorical_features = ["location_type"]

numeric_features = [
    "financial_aid_status",
    "internet_connectivity_issues",
    "motivation_score",
    "career_alignment",
    "stress_levels",
    "family_support",
    "attendance_rate",
    "test_scores_avg",
    "backlogs",
    "teaching_quality_rating"
]

In [4]:
log_transformer = Pipeline(steps=[
    ("log", FunctionTransformer(np.log1p, validate=False)),
    ("scaler", StandardScaler())
])

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("log", log_transformer, log_features),
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features)
    ]
)

In [6]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

In [7]:
df = pd.read_csv("dropout_dataset_clean.csv")

X = df.drop("dropout", axis=1)
y = df["dropout"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5600, 13)
Test shape: (1400, 13)


In [9]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('log',
                                                  Pipeline(steps=[('log',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['family_income',
                                                   'distance_to_institute']),
                                                 ('num', StandardScaler(),
                                                  ['financial_aid_status',
                                                   'internet_connectivity_issues',
                                                   'motivation_score',
                                                   'career_alignment',
                                                   'stress_levels',
                                                   'family_support',
                                                   'attendance_rate',
                                                   'test_scores_avg',
                                                   'backlogs',
                                                   'teaching_quality_rating']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['location_type'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [10]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.86      0.90       800
           1       0.84      0.92      0.87       600

    accuracy                           0.89      1400
   macro avg       0.88      0.89      0.89      1400
weighted avg       0.89      0.89      0.89      1400

Confusion Matrix:
[[692 108]
 [ 50 550]]
ROC-AUC Score: 0.9673458333333333


In [11]:
with open("staylearn_model.pkl", "wb") as f:
  pickle.dump(pipeline, f)

In [12]:
joblib.dump(pipeline, "staylearn_model.joblib")

['staylearn_model.joblib']

In [13]:
loaded_model = joblib.load("staylearn_model.joblib")
print("model berhasil di-load")

model berhasil di-load
